# 04 · Build analytic base

Este notebook audita a base analítica integrada construída para a fase final da análise.
Ele verifica cobertura horária do clima, flags de qualidade da mobilidade, reconciliação
de rotas e a janela integrada de 19 dias usada nos modelos que combinam ticket + clima + mobilidade.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 120})


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / 'data' / 'derived').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root from the current working directory.')


PROJECT_ROOT = locate_project_root()
DERIVED = PROJECT_ROOT / 'data' / 'derived'
FIGURES = DERIVED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
WEATHER_ORDER = ['Clear', 'Light Rain', 'Moderate Rain', 'Heavy Rain / Storm']


def savefig(name: str) -> Path:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path


In [ ]:
weather = pd.read_parquet(DERIVED / 'weather_hourly.parquet')
ticket = pd.read_parquet(DERIVED / 'ticket_hourly_route_profile.parquet')
mobility = pd.read_parquet(DERIVED / 'mobility_hourly_route_direction.parquet')
integrated = pd.read_parquet(DERIVED / 'integrated_route_hour.parquet')
quality = pd.read_csv(DERIVED / 'mobility_day_quality_flags.csv', parse_dates=['date'])
crosswalk = pd.read_csv(DERIVED / 'route_crosswalk.csv')
trip_coverage = pd.read_csv(DERIVED / 'trip_base_coverage.csv')

print('=== Deliverables ===')
print('weather_hourly rows:', len(weather))
print('ticket_hourly_route_profile rows:', len(ticket))
print('mobility_hourly_route_direction rows:', len(mobility))
print('integrated_route_hour rows:', len(integrated))
print()

print('=== Weather hourly completeness ===')
print('hour keys duplicated:', int(weather[['date', 'hour']].duplicated().sum()))
print('missing observation rows:', int(weather['weather_observation_missing'].sum()))
print(weather.loc[weather['weather_observation_missing'], ['date', 'hour']].to_string(index=False))
print()

print('=== Ticket coverage ===')
print('ticket days:', pd.to_datetime(ticket['date']).dt.date.nunique())
print('profiles:', sorted(ticket['card_label'].dropna().astype(str).unique().tolist()))
print('boardings sum:', int(ticket['boardings'].sum()))


In [ ]:
print('=== Mobility quality flags ===')
print(quality[['date', 'row_count', 'row_count_ratio', 'is_partial_day', 'partial_day_reason']].to_string(index=False))
print()

print('=== Route crosswalk resolution ===')
print(crosswalk['route_crosswalk_confidence'].value_counts(dropna=False).to_string())
print()
print('Unresolved routes excluded from integrated models:')
print(crosswalk.loc[crosswalk['manual_review'], ['route_norm', 'rationale']].drop_duplicates().to_string(index=False))
print()

print('=== GTFS trip-base coverage ===')
matched = int(trip_coverage['in_gtfs'].sum())
total = int(len(trip_coverage))
print(f'matched trip bases: {matched}/{total}')
print('unmatched trip bases:', sorted(trip_coverage.loc[~trip_coverage['in_gtfs'], 'trip_base'].tolist()))


In [ ]:
print('=== Integrated 19-day window ===')
integrated_dates = pd.to_datetime(integrated['date'])
print('unique integrated days:', integrated_dates.dt.date.nunique())
print('min date:', integrated_dates.min())
print('max date:', integrated_dates.max())
print('route_crosswalk_confidence present:', 'route_crosswalk_confidence' in integrated.columns)
print('coverage_flag present:', 'coverage_flag' in integrated.columns)
print()

print('Integrated confidence mix:')
print(integrated['route_crosswalk_confidence'].value_counts(dropna=False).to_string())
print()

print('Rows with mobility metrics present:', int(integrated['observed_trip_count'].notna().sum()))
print('Rows without mobility metrics present:', int(integrated['observed_trip_count'].isna().sum()))
print()

print('Integrated sample:')
print(integrated.head(12).to_string(index=False))
